# WM-811K — export of predicted probabilities under the transformation group

This notebook loads every fitted classifier listed in `CFG.models` (three or
more) and stores, for each, its predicted probabilities for every held-out wafer
under every element of the dihedral group $D_4$, together with the diagnostics
computed here (dispersion along the orbit, held-out metrics) and a manifest of
everything written. The conformal analysis is carried out in a separate
notebook, which reads these arrays.

Splitting the work this way is what makes the study cheap. The classifier is
fitted once and held fixed, so the eight forward passes are done once here; the
conformal notebook then resamples calibration/test splits of an array already in
memory. Recomputing the network inside the replication loop would multiply the
cost by the number of replications for no gain.

**The transformation group.** $D_4$ consists of the four rotations by multiples
of 90 degrees and the four reflections obtained by composing them with a
horizontal flip. The wafer maps are square, so every element maps the pixel grid
onto itself exactly: no interpolation, no padding, no loss at the borders. The
group has eight elements, so the orbit average is computed exactly rather than
approximated by sampling.

Invariance is only approximate here, and deliberately so. Edge-Ring, Center,
Donut, Random and Near-full are defined by radial structure or by the absence of
directional structure, so a rotation or reflection plausibly preserves the
class; Edge-Loc, Loc and Scratch are defined by a linear trace or a localised
region, and for those the class-conditional distribution of orientations need
not be invariant. Conformal validity does not require invariance, so the case
study tests whether the departure shows up in efficiency rather than in
coverage.

## 1. Setup

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA PyTorch:", torch.version.cuda)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Architetture supportate:", torch.cuda.get_arch_list())

PyTorch: 2.10.0+cu128
CUDA PyTorch: 12.8
CUDA disponibile: True
GPU: Tesla T4
Compute capability: (7, 5)
Architetture supportate: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


## 2. Libraries

In [2]:
import os
import sys
import copy
import glob
import hashlib
import json
import math
import time
import random
import shutil
import datetime
import warnings
from contextlib import contextmanager

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from albumentations import Compose, Normalize, Resize
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             recall_score, roc_auc_score)

# Locate the utilities among the datasets attached to the notebook.
matches = glob.glob("/kaggle/input/**/general_utils.py", recursive=True)
if not matches:
    raise FileNotFoundError("general_utils.py non trovato sotto /kaggle/input")
UTILS_DIR = os.path.dirname(matches[0])
if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)
print("Cartella delle utilities:", UTILS_DIR)

from general_utils import image_path_generation, get_model_probs

# Deliberately no `from effnet_utils import *`: it redefines get_transforms,
# TrainDataset and the configuration, and re-running it after section 3 silently
# replaces the definitions below (the cause of "Input height (404) doesn't match
# model (224)"). Nothing from it is needed for the export.

Cartella delle utilities: /kaggle/input/datasets/alaurenzi/wm811k-utils


## 3. Configuration and model utilities

The dataset, the preprocessing and the model wrapper are taken unchanged from
the training notebook, so that each network sees exactly the input it was
trained on. The training loop is not needed here and is not included.

In [3]:
# ====================================================
# Configuration
# ====================================================

class CFG:
    print_freq=100
    num_workers = 4
    # Default only; the export loop sets model_name for each entry of `models`.
    model_name = 'tf_efficientnet_b0_ns'   #'tf_efficientnet_b2_ns' #'vgg16' #'resnext50_32x4d' #'tf_efficientnet_l2_ns_475'  #'tf_efficientnet_b2_ns' #'resnext50_32x4d'  #'coat_tiny'   
    #output_dir = '/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/WMDD_application/models'
    #intervalplot_dir = '/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/WMDD_application/resuls'
    output_dir='./'
    intervalplot_dir = './'
    size = 101 #712
    epochs = 25 # 100
    factor = 0.2
    patience = 3
    eps = 1e-6
    lr = 1e-4
    min_lr = 1e-6
    batch_size = 16
    weight_decay = 1e-6
    gradient_accumulation_steps = 1
    max_grad_norm = 1000
    seed = 42
    target_size = 8
    target_col = 'labels'
    n_fold = 3
    #trn_fold = [1,2,3,4,5]
    trn_fold = [0,1,2]
    score_plot = True
    img_size = 224
    #device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    label2int = {
    'Center': 0, 'Donut': 1, 'Edge-Loc': 2, 'Edge-Ring': 3,
    'Loc': 4, 'Random': 5, 'Scratch': 6, 'Near-full': 7
                                                        }
    randomize = True, #for  get_APS_scores_all function
    seed = 0         #for  get_APS_scores_all function
    n_sim = 1000
    alpha=0.01
    score = 'APS'  # method to compute scores ('APS', 'RAPS', 'STD')
    numclasses = 8

    # ------------------------------------------------------------------
    # Export
    # ------------------------------------------------------------------
    # Models to export, any number. Each needs a checkpoint f'{name}_best.pth'.
    models = ['tf_efficientnet_b0_ns', 'resnext50_32x4d', 'coat_tiny']
    # Where checkpoints are looked for. None: anywhere under /kaggle/input.
    # A model found in more than one place stops the notebook: then list here
    # the directory of the training run to use.
    ckpt_dirs = None
    img_size_override = {}        # {name: size} for a model trained at another size
    file_tag = {}                 # {name: prefix}; by default the prefix is the model name
    out_dir = '/kaggle/working/csda_revision/data'             # arrays read by the conformal notebook
    tables_dir = '/kaggle/working/csda_revision/export_tables' # diagnostics of this notebook
    export_workers = 2            # DataLoader workers; 0 if worker processes cause trouble
    overwrite = False             # False: an export made from the same checkpoint and held-out set is reloaded

cfg=CFG()

In [4]:
# ====================================================
# Dataset and transforms (as in the training notebook)
# ====================================================
class TrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.file_names = df['image_id'].values
        self.labels = df['labels'].values
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = cv2.imread(self.file_names[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        label = torch.tensor(self.labels[idx]).long()
        return image, label

def get_transforms(data, cfg):
    transform_list = [
        Resize(cfg.img_size, cfg.img_size),
        Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]
    return Compose(transform_list)


# ====================================================
# Model definition (as in the training notebook)
# ====================================================
class Model_type(nn.Module):
    #def __init__(self, model_name='tf_efficientnet_b2_ns', pretrained=False, cfg=None):
    #    super().__init__()
    #def __init__(self, model_name='resnext50_32x4d_best.pth', pretrained=False, cfg=None):
     #   super().__init__()
    def __init__(self, model_name=cfg.model_name, pretrained=False, cfg=None):
        super().__init__()

        # If cfg is not provided, fallback to a default
        if cfg is None:
            cfg = CFG()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=cfg.target_size)


    def forward(self, x):
        x = self.model(x)
        return x

## 4. The transformation group

The group element is applied to the raw image, before normalisation, so that
the transformation and the preprocessing do not interact.

In [5]:
# ---------------------------------------------------------------------------
# Dihedral group D4 applied inside the dataset pipeline.
#
# The transformation acts on the raw image, before resizing and normalisation.
# The images are square (404 x 404), so rotations by multiples of 90 degrees map
# the pixel grid onto itself exactly: no interpolation, no padding, no loss at
# the borders.
# ---------------------------------------------------------------------------

from image_groups import DihedralGroup

GROUP = DihedralGroup()               # 8 elements; the full orbit is enumerable

class TransformedDataset(TrainDataset):
    """TrainDataset with one fixed group element applied to every image."""

    def __init__(self, df, g, transform=None):
        super().__init__(df, transform)
        self.g = int(g)

    def __getitem__(self, idx):
        image = cv2.imread(self.file_names[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # apply expects a batch, so add and remove the leading axis
        image = GROUP.apply(image[None, ...], self.g)[0]
        if self.transform:
            image = self.transform(image=image)['image']
        return image, torch.tensor(self.labels[idx]).long()


def orbit_probs(model, df, device, cfg, group=GROUP):
    """Predicted probabilities under every group element: shape (|G|, n, K).

    Computed once and saved. The conformal notebook then resamples
    calibration/test splits of an array already in memory; recomputing the
    network inside the replication loop would multiply the cost by R for
    nothing, since the model is fixed.
    """
    out = []
    for g in group.full_orbit():
        ds = TransformedDataset(df, g, transform=get_transforms(data='valid', cfg=cfg))
        dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.export_workers, pin_memory=True)
        out.append(get_model_probs(model, dl, device))
    return np.stack(out)


def plain_probs(model, df, device, cfg):
    """Predicted probabilities with no group element applied: shape (n, K)."""
    ds = TrainDataset(df, transform=get_transforms(data='valid', cfg=cfg))
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                    num_workers=cfg.export_workers, pin_memory=True)
    return get_model_probs(model, dl, device)

## 5. Data

Calibration and test are merged into a single held-out set. The conformal
analysis resamples the calibration/test split at every replication, so the
variability of calibration is measured rather than fixed by one arbitrary
split. The training and validation images are not touched: the model must not
have seen any wafer used for calibration or evaluation.

The relative image paths of the held-out set, in export order, are saved once
as `heldout_index.csv`, and their SHA-1 is stored in the metadata of every
model. Two exports refer to the same wafers in the same order exactly when the
hashes agree, which the labels alone cannot establish.

In [6]:
#dataset_path = f'{BASE}/WMDD_application/dataset/'
dataset_path = '/kaggle/input/datasets/alaurenzi/wm811k-images-dataset/images/'

df = pd.read_csv(dataset_path + 'WM_811k_subset.csv', index_col=0)
df['image_id'] = df.apply(lambda row: image_path_generation(row, base_path=dataset_path), axis=1)

heldout = (df[df['set'].isin(['cal', 'test'])][['labels', 'image_id']]
             .reset_index(drop=True))
heldout['labels'] = heldout['labels'].map(cfg.label2int)

assert heldout['labels'].notna().all(), "some label is missing from cfg.label2int"
heldout['labels'] = heldout['labels'].astype(int)

INT2LABEL = {v: k for k, v in cfg.label2int.items()}
labels = heldout['labels'].to_numpy().astype(np.int64)

# Identity of the held-out set: relative paths, independent of where the dataset is mounted.
HELDOUT_IDS = [os.path.relpath(p, dataset_path) for p in heldout['image_id']]
HELDOUT_SHA1 = hashlib.sha1('\n'.join(HELDOUT_IDS).encode()).hexdigest()

os.makedirs(cfg.out_dir, exist_ok=True)
os.makedirs(cfg.tables_dir, exist_ok=True)
pd.DataFrame({'image_id': HELDOUT_IDS, 'label': labels,
              'class': [INT2LABEL[k] for k in labels]}).to_csv(f'{cfg.out_dir}/heldout_index.csv', index=False)

print('held-out size:', heldout.shape, '| sha1:', HELDOUT_SHA1[:12])
display(heldout['labels'].map(INT2LABEL).value_counts().rename('wafers').to_frame().T)

held-out size: (3649, 2) | sha1: 32ccb8fe45da


labels,Edge-Ring,Edge-Loc,Center,Loc,Random,Scratch,Donut,Near-full
wafers,1389,635,615,413,203,202,148,44


## 6. Models

All checkpoints are located, and every model is checked, before any prediction
is computed. A model whose checkpoint is missing, or found in more than one
place (for instance in an older model dataset and in the output of the new
training run), stops the notebook: the checkpoint used must be a choice, not
whichever the search happens to return first.

The weights are loaded with `strict=True`: with `strict=False` a checkpoint
whose keys do not match the architecture loads nothing, and the notebook would
export the predictions of a randomly initialised network without any error.

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def tag_of(name):
    """File prefix of a model in cfg.out_dir."""
    return cfg.file_tag.get(name, name)


def model_cfg(name, base=cfg):
    """Copy of the configuration with the model-specific fields set."""
    c = copy.copy(base)
    c.model_name = name
    c.img_size = base.img_size_override.get(name, base.img_size)
    return c


def find_checkpoint(name, dirs):
    fname = f'{name}_best.pth'
    if dirs is None:
        hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    else:
        hits = [os.path.join(d, fname) for d in dirs if os.path.isfile(os.path.join(d, fname))]
    hits = sorted({os.path.realpath(p) for p in hits})      # the same file reached through two mount paths counts once
    if len(hits) != 1:
        where = 'under /kaggle/input' if dirs is None else f'in {dirs}'
        hint = ' Set CFG.ckpt_dirs to the directory of the training run to use.' if len(hits) > 1 else ''
        raise FileNotFoundError(f'{name}: expected exactly one {fname} {where}, found {hits}.{hint}')
    return hits[0]


def training_record(ckpt):
    """The _done.json written by the training notebook next to the checkpoint, if any."""
    f = ckpt.removesuffix('_best.pth') + '_done.json'
    if not os.path.isfile(f):
        return None
    with open(f) as fh:
        return json.load(fh)


def load_model(name, c):
    state = torch.load(CKPT[name], map_location='cpu', weights_only=False)['model']
    state = {k.removeprefix('module.'): v for k, v in state.items()}   # DataParallel checkpoints
    model = Model_type(name, pretrained=False, cfg=c)
    model.load_state_dict(state, strict=True)
    return model.to(device).eval()


def preflight(model, c, df):
    """Input size and head size agree with the pipeline, for every group element."""
    s = c.img_size
    with torch.no_grad():
        k = model(torch.zeros(1, 3, s, s, device=device)).shape[1]
    assert k == c.numclasses, f'{c.model_name}: head has {k} outputs, expected {c.numclasses}'
    tf = get_transforms(data='valid', cfg=c)
    for g in GROUP.full_orbit():
        x, _ = TransformedDataset(df.iloc[:1], g, transform=tf)[0]
        assert tuple(x.shape) == (3, s, s), \
            f'{c.model_name}, g={g}: pipeline gives {tuple(x.shape)}, model expects (3, {s}, {s})'


# Every checkpoint must be found exactly once and every file tag must be distinct.
assert len(cfg.models) == len(set(cfg.models)), f'duplicate names in CFG.models: {cfg.models}'
CKPT, problems = {}, []
for name in cfg.models:
    try:
        CKPT[name] = find_checkpoint(name, cfg.ckpt_dirs)
    except FileNotFoundError as e:
        problems.append(str(e))
if problems:
    raise FileNotFoundError('\n'.join(problems))

tags = [tag_of(n) for n in cfg.models]
assert len(set(tags)) == len(tags), f'two models share a file tag: {tags}'

w = max(map(len, cfg.models))
for name in cfg.models:
    rec = training_record(CKPT[name])
    info = (f"val AUC {rec['best_val_auc']:.4f}, epoch {rec['best_epoch']}/{rec['epochs_run']}"
            if rec else 'no _done.json next to the checkpoint')
    print(f'{name:{w}s}  {CKPT[name]}  ({info})')
print('device:', device)

tf_efficientnet_b0_ns  /kaggle/input/datasets/alaurenzi/wm811k-train-model/tf_efficientnet_b0_ns_best.pth  (val AUC 0.9937, epoch 11/15)
resnext50_32x4d        /kaggle/input/datasets/alaurenzi/wm811k-train-model/resnext50_32x4d_best.pth  (val AUC 0.9945, epoch 8/12)
coat_tiny              /kaggle/input/datasets/alaurenzi/wm811k-train-model/coat_tiny_best.pth  (val AUC 0.9971, epoch 4/8)
device: cuda


## 7. Export

Three assertions guard the output. The shape must match the group size, the
number of wafers and the number of classes; the rows must be probability
vectors; and element 0 of the group, being the identity, must reproduce the
untransformed prediction. The last one is the check that the ordering of the
group elements agrees with what the conformal notebook assumes.

For each model the loop writes `{tag}_probs.npy`, `{tag}_labels.npy` and
`{tag}_meta.json` to `CFG.out_dir`. With `overwrite = False` an existing export
is reloaded only if its metadata records the same checkpoint and the same
held-out hash; otherwise it is recomputed, so a stale array from an earlier
checkpoint cannot be carried forward.

In [8]:
def reusable_export(name, tag):
    """Existing export built from the same checkpoint and the same held-out set, or None."""
    f_probs, f_meta = f'{cfg.out_dir}/{tag}_probs.npy', f'{cfg.out_dir}/{tag}_meta.json'
    if cfg.overwrite or not (os.path.isfile(f_probs) and os.path.isfile(f_meta)):
        return None
    with open(f_meta) as fh:
        meta = json.load(fh)
    if meta.get('checkpoint') != CKPT[name] or meta.get('heldout_sha1') != HELDOUT_SHA1:
        print('existing export refers to another checkpoint or held-out set: recomputed')
        return None
    probs = np.load(f_probs)
    assert probs.shape == (GROUP.size, len(labels), cfg.numclasses), f'{f_probs}: shape {probs.shape}'
    assert np.array_equal(np.load(f'{cfg.out_dir}/{tag}_labels.npy'), labels), f'{tag}: labels differ'
    return probs


PROBS = {}
for name in cfg.models:
    c = model_cfg(name)
    tag = tag_of(name)
    print(f'\n=== {name} (img_size={c.img_size}) -> {tag} ===')

    probs = reusable_export(name, tag)
    if probs is not None:
        PROBS[name] = probs
        print(f'already exported from this checkpoint, reloaded {probs.shape}')
        continue

    model = load_model(name, c)
    preflight(model, c, heldout)

    probs = orbit_probs(model, heldout, device, c)          # (|G|, n, K)
    plain = plain_probs(model, heldout, device, c)          # (n, K)

    assert probs.shape == (GROUP.size, len(labels), c.numclasses)
    assert np.allclose(probs.sum(axis=2), 1.0, atol=1e-4), "rows must sum to one"
    assert np.allclose(probs[0], plain, atol=1e-4), "element 0 is not the identity"

    probs = probs.astype(np.float32)
    acc = float((probs[0].argmax(1) == labels).mean())
    np.save(f'{cfg.out_dir}/{tag}_probs.npy', probs)
    np.save(f'{cfg.out_dir}/{tag}_labels.npy', labels)
    with open(f'{cfg.out_dir}/{tag}_meta.json', 'w') as fh:
        json.dump({'model_name': name, 'file_tag': tag, 'checkpoint': CKPT[name],
                   'training_run': training_record(CKPT[name]),
                   'img_size': c.img_size, 'group': type(GROUP).__name__,
                   'group_size': int(GROUP.size), 'n': int(len(labels)),
                   'numclasses': int(c.numclasses), 'label2int': c.label2int,
                   'heldout_sha1': HELDOUT_SHA1, 'heldout_accuracy': acc,
                   'timm': timm.__version__, 'torch': torch.__version__,
                   'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
                   'exported': datetime.datetime.now().isoformat(timespec='seconds')},
                  fh, indent=2)

    PROBS[name] = probs
    print(f'saved {probs.shape} to {cfg.out_dir}; held-out accuracy {acc:.3f}')

    del model, plain
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\nexported:', list(PROBS))


=== tf_efficientnet_b0_ns (img_size=224) -> tf_efficientnet_b0_ns ===


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c5b0059ae80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c5b0059ae80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

saved (8, 3649, 8) to /kaggle/working/csda_revision/data; held-out accuracy 0.935

exported: ['tf_efficientnet_b0_ns', 'resnext50_32x4d', 'coat_tiny']


## 8. Dispersion along the orbit

How much the predicted probabilities move as the group element varies. The
theory says orbit averaging helps to the extent that the score varies along the
orbit, so this is the quantity to look at before any calibration. If the radial
classes disperse less than the directional ones, the approximate nature of the
invariance is already visible here.

One column per model. Both tables are saved to `CFG.tables_dir`.

In [9]:
RADIAL = ['Center', 'Donut', 'Edge-Ring', 'Random', 'Near-full']

rows = []
for name, probs in PROBS.items():
    disp = probs.std(axis=0).mean(axis=1)          # per wafer, across group elements
    for k in range(cfg.numclasses):
        m = labels == k
        cls = INT2LABEL[k]
        rows.append({'model': name, 'class': cls,
                     'symmetry': 'radial' if cls in RADIAL else 'directional',
                     'n': int(m.sum()), 'dispersion': float(disp[m].mean())})

dispersion = pd.DataFrame(rows)
disp_by_class = dispersion.pivot_table(index=['symmetry', 'class', 'n'], columns='model',
                                       values='dispersion')[list(PROBS)]
disp_by_symmetry = dispersion.groupby(['symmetry', 'model'])['dispersion'].mean().unstack('model')[list(PROBS)]

dispersion.to_csv(f'{cfg.tables_dir}/dispersion_long.csv', index=False)
disp_by_class.to_csv(f'{cfg.tables_dir}/dispersion_by_class.csv')
disp_by_symmetry.to_csv(f'{cfg.tables_dir}/dispersion_by_symmetry.csv')

display(disp_by_class.round(4))
display(disp_by_symmetry.round(4))

model                       tf_efficientnet_b0_ns  resnext50_32x4d  coat_tiny
symmetry    class     n                                                      
directional Edge-Loc  635                  0.0309           0.0421     0.0214
            Loc       413                  0.0584           0.0460     0.0201
            Scratch   202                  0.0412           0.0330     0.0248
radial      Center    615                  0.0238           0.0100     0.0037
            Donut     148                  0.0287           0.0305     0.0248
            Edge-Ring 1389                 0.0041           0.0047     0.0017
            Near-full 44                   0.0124           0.0122     0.0021
            Random    203                  0.0169           0.0135     0.0161

model,tf_efficientnet_b0_ns,resnext50_32x4d,coat_tiny
symmetry,,,
directional,0.0435,0.0404,0.0221
radial,0.0172,0.0142,0.0097


## 9. Held-out metrics of the fitted models

Overall and per-class metrics on the held-out set, from the untransformed
predictions (group element 0), to confirm that the loaded weights reproduce the
performance recorded at training time. The orbit-averaged columns use the mean
of the eight probability vectors. AUCs are one-vs-rest and computed from the
probabilities. All tables are saved to `CFG.tables_dir`.

In [10]:
overall, per_class = [], []
for name, probs in PROBS.items():
    p, p_avg = probs[0], probs.mean(axis=0)
    yhat, yhat_avg = p.argmax(axis=1), p_avg.argmax(axis=1)
    overall.append({'model': name,
                    'accuracy': accuracy_score(labels, yhat),
                    'macro_recall': recall_score(labels, yhat, average='macro'),
                    'macro_f1': f1_score(labels, yhat, average='macro'),
                    'auc_ovr_macro': roc_auc_score(labels, p, multi_class='ovr', average='macro'),
                    'accuracy_orbit_avg': accuracy_score(labels, yhat_avg),
                    'macro_recall_orbit_avg': recall_score(labels, yhat_avg, average='macro')})

    ks = np.arange(cfg.numclasses)
    prec, rec, f1, sup = precision_recall_fscore_support(labels, yhat, labels=ks, zero_division=0)
    for k in ks:
        per_class.append({'model': name, 'class': INT2LABEL[k], 'n': int(sup[k]),
                          'precision': prec[k], 'recall': rec[k], 'f1': f1[k],
                          'auc_ovr': roc_auc_score(labels == k, p[:, k])})

overall = pd.DataFrame(overall).set_index('model')
per_class = pd.DataFrame(per_class)

overall.to_csv(f'{cfg.tables_dir}/heldout_metrics.csv')
per_class.to_csv(f'{cfg.tables_dir}/heldout_metrics_per_class.csv', index=False)

display(overall.round(4))
for metric in ['recall', 'auc_ovr']:
    print(metric)
    display(per_class.pivot_table(index=['class', 'n'], columns='model', values=metric)[list(PROBS)].round(3))

,accuracy,macro_recall,macro_f1,auc_ovr_macro,accuracy_orbit_avg,macro_recall_orbit_avg
model,,,,,,
tf_efficientnet_b0_ns,0.9186,0.8922,0.8882,0.9935,0.9315,0.9131
resnext50_32x4d,0.9235,0.9046,0.9023,0.9927,0.9287,0.9178
coat_tiny,0.9351,0.9140,0.9111,0.9964,0.9389,0.9150


recall


,model,tf_efficientnet_b0_ns,resnext50_32x4d,coat_tiny
class,n,,,
Center,615,0.946,0.979,0.985
Donut,148,0.905,0.872,0.899
Edge-Loc,635,0.902,0.850,0.874
Edge-Ring,1389,0.991,0.982,0.988
Loc,413,0.697,0.792,0.862
Near-full,44,0.932,0.955,1.000
Random,203,0.926,0.931,0.852
Scratch,202,0.837,0.876,0.851


auc_ovr


,model,tf_efficientnet_b0_ns,resnext50_32x4d,coat_tiny
class,n,,,
Center,615,0.997,0.998,0.999
Donut,148,0.997,0.993,0.998
Edge-Loc,635,0.990,0.984,0.994
Edge-Ring,1389,0.999,0.999,0.999
Loc,413,0.973,0.977,0.986
Near-full,44,1.000,1.000,1.000
Random,203,0.999,0.998,0.998
Scratch,202,0.992,0.993,0.998


## 10. Manifest and archive

`manifest.json` in `CFG.out_dir` lists the models of this run with their
checkpoints and files, the held-out hash and the tables written. Any other
`*_probs.npy` found in `CFG.out_dir` (from an earlier run in the same session)
is reported, because a conformal notebook that discovers models automatically
would load it. Everything under `csda_revision/` is then zipped into a single
file to download or to keep in the notebook output.

In [11]:
exported_tags = {tag_of(n) for n in PROBS}
stray = sorted(os.path.basename(f).removesuffix('_probs.npy')
               for f in glob.glob(f'{cfg.out_dir}/*_probs.npy')
               if os.path.basename(f).removesuffix('_probs.npy') not in exported_tags)
if stray:
    print(f'WARNING: {cfg.out_dir} also contains exports not produced by this run: {stray}')

manifest = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'models': {n: {'file_tag': tag_of(n), 'checkpoint': CKPT[n],
                   'files': [f'{tag_of(n)}_probs.npy', f'{tag_of(n)}_labels.npy', f'{tag_of(n)}_meta.json']}
               for n in PROBS},
    'heldout': {'n': int(len(labels)), 'sha1': HELDOUT_SHA1, 'index': 'heldout_index.csv'},
    'group': {'name': type(GROUP).__name__, 'size': int(GROUP.size)},
    'tables_dir': cfg.tables_dir,
    'tables': sorted(os.listdir(cfg.tables_dir)),
    'stray_exports': stray,
    'timm': timm.__version__, 'torch': torch.__version__,
}
with open(f'{cfg.out_dir}/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)

root = os.path.dirname(cfg.out_dir.rstrip('/'))          # /kaggle/working/csda_revision
archive = shutil.make_archive(root + '_export', 'zip',
                              root_dir=os.path.dirname(root), base_dir=os.path.basename(root))
print(f'manifest: {cfg.out_dir}/manifest.json')
print(f'archive:  {archive} ({os.path.getsize(archive) / 2**20:.1f} MB)')

manifest: /kaggle/working/csda_revision/data/manifest.json
archive:  /kaggle/working/csda_revision_export.zip (2.5 MB)
